# Qwen2.5-0.5B CPU Baseline  Sequential LoRA

A self-contained Google Colab notebook for a **lightweight CPU baseline**. It uses **Qwen2.5-0.5B-Instruct + LoRA**, trains sequentially on 8 small continual-learning tasks, evaluates every task after every stage, and reports **OP, BWT, and mean forgetting**.

**Important:** this is a CPU-friendly baseline/sanity benchmark. It uses tiny built-in task samples so it can run without downloading external benchmark datasets. Therefore its numbers are **not directly comparable** to the full 8-task TRACE benchmark unless the same task data/preprocessing are substituted.


In [16]:
!pip -q install -U transformers peft accelerate datasets sentencepiece torchao

In [8]:
import os, json, random, math, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cpu')
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
OUT = Path('/content/qwen05b_baseline_results')
OUT.mkdir(exist_ok=True)
print('Device:', DEVICE)
print('Model:', MODEL_NAME)


Device: cpu
Model: Qwen/Qwen2.5-0.5B-Instruct


## 1. Tiny continual-learning task suite

The task names mirror the project structure. Each task contains short instruction/answer examples so the notebook remains CPU-runnable and deterministic. Replace the examples with the exact benchmark examples later if you need an apples-to-apples comparison.

In [9]:
TASKS = {
 'C-STANCE': [
  ('Classify the stance: The speaker supports stricter climate policy.', 'support'),
  ('Classify the stance: The speaker opposes the proposed climate tax.', 'oppose'),
  ('Classify the stance: The speaker gives no clear position.', 'neutral'),
 ],
 'FOMC': [
  ('What is the main goal of monetary policy when inflation is too high?', 'reduce inflation'),
  ('What happens to borrowing costs when policy rates rise?', 'borrowing becomes more expensive'),
  ('What does a rate cut generally encourage?', 'more borrowing and spending'),
 ],
 'MeetingBank': [
  ('Summarize: The committee approved the previous minutes and scheduled the next review.', 'The committee approved the minutes and scheduled the next review.'),
  ('Summarize: Members discussed budget pressure and agreed to monitor spending monthly.', 'Members discussed budget pressure and agreed to monitor spending monthly.'),
  ('Summarize: The chair asked the team to prepare a revised project timeline.', 'The chair requested a revised project timeline.'),
 ],
 'Py150': [
  ('Complete Python: def add(a, b): return ____', 'a + b'),
  ('Complete Python: items = [1,2,3]; print(len(____))', 'items'),
  ('Complete Python: for x in nums: ____ x', 'print'),
 ],
 'ScienceQA': [
  ('What gas do plants absorb during photosynthesis?', 'carbon dioxide'),
  ('What force pulls objects toward Earth?', 'gravity'),
  ('What is H2O commonly called?', 'water'),
 ],
 'NumGLUE-cm': [
  ('Calculate: 12 + 8', '20'),
  ('Calculate: 7 * 6', '42'),
  ('Calculate: 81 / 9', '9'),
 ],
 'NumGLUE-ds': [
  ('A box has 5 rows of 4 items. How many items?', '20'),
  ('A $50 item is discounted by 10%. What is the price?', '45'),
  ('A car travels 60 km in 2 hours. What is its average speed?', '30 km/h'),
 ],
 '20Minuten': [
  ('Write a one-sentence news-style summary: The city opened a new public library today.', 'The city opened a new public library today.'),
  ('Write a one-sentence news-style summary: Heavy rain delayed several trains.', 'Heavy rain delayed several trains.'),
  ('Write a one-sentence news-style summary: Researchers announced a new battery prototype.', 'Researchers announced a new battery prototype.'),
 ]
}
task_names = list(TASKS)
print('Tasks:', task_names)


Tasks: ['C-STANCE', 'FOMC', 'MeetingBank', 'Py150', 'ScienceQA', 'NumGLUE-cm', 'NumGLUE-ds', '20Minuten']


In [10]:
import os
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model
from huggingface_hub import login

# Automatically log in if HF_TOKEN is configured in Colab Secrets to resolve the warning
try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        login(token=hf_token, add_to_git_credential=False)
        print("Successfully authenticated with Hugging Face Hub.")
except Exception:
    pass

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
).to(DEVICE)

lora_config = LoraConfig(
    r=4,
    lora_alpha=8,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


## 2. CPU-safe training utilities

This deliberately avoids `bitsandbytes`/4-bit quantization because the goal is a straightforward CPU run. The baseline is therefore **LoRA**, rather than GPU-oriented QLoRA.

In [11]:
from torch.optim import AdamW

MAX_LEN = 192
EPOCHS = 1
LR = 2e-4

def format_example(question, answer):
    messages = [
        {'role':'system','content':'Answer briefly and directly.'},
        {'role':'user','content':question},
        {'role':'assistant','content':answer},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

def train_task(task_name):
    model.train()
    optimizer = AdamW((p for p in model.parameters() if p.requires_grad), lr=LR)
    losses = []
    examples = TASKS[task_name]
    for epoch in range(EPOCHS):
        order = list(range(len(examples)))
        random.shuffle(order)
        for i in order:
            text = format_example(*examples[i])
            enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_LEN)
            input_ids = enc['input_ids'].to(DEVICE)
            attention_mask = enc['attention_mask'].to(DEVICE)
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
            loss = out.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); optimizer.zero_grad(set_to_none=True)
            losses.append(float(loss.detach()))
    return float(np.mean(losses))

@torch.no_grad()
def score_task(task_name):
    # Exact-answer substring accuracy; deliberately simple and robust for CPU.
    model.eval()
    correct = 0
    for question, answer in TASKS[task_name]:
        prompt = tokenizer.apply_chat_template(
            [{'role':'system','content':'Answer briefly and directly.'},
             {'role':'user','content':question}],
            tokenize=False, add_generation_prompt=True)
        enc = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=MAX_LEN).to(DEVICE)
        gen = model.generate(**enc, max_new_tokens=24, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        new_tokens = gen[0, enc['input_ids'].shape[1]:]
        pred = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
        if answer.lower() in pred or pred in answer.lower():
            correct += 1
    return correct / len(TASKS[task_name])


## 3. Sequential baseline run

At stage *t*, the model trains only on task *t*, then evaluates on **all tasks**. This creates the performance matrix needed for continual-learning analysis.

In [12]:
performance = np.zeros((len(task_names), len(task_names)), dtype=float)
stage_losses = []
start = time.time()

for t, task in enumerate(task_names):
    loss = train_task(task)
    stage_losses.append(loss)
    print(f'\nStage {t+1}/{len(task_names)}: {task} | train loss={loss:.4f}')
    for j, eval_task in enumerate(task_names):
        performance[t, j] = score_task(eval_task)
        print(f'  {eval_task:12s}: {performance[t,j]:.3f}')

print(f'\nTotal runtime: {(time.time()-start)/60:.1f} minutes')



Stage 1/8: C-STANCE | train loss=5.6845
  C-STANCE    : 1.000
  FOMC        : 0.000
  MeetingBank : 0.000
  Py150       : 0.667
  ScienceQA   : 0.667
  NumGLUE-cm  : 1.000
  NumGLUE-ds  : 0.667
  20Minuten   : 0.000

Stage 2/8: FOMC | train loss=4.3582
  C-STANCE    : 1.000
  FOMC        : 0.000
  MeetingBank : 0.000
  Py150       : 0.667
  ScienceQA   : 0.667
  NumGLUE-cm  : 1.000
  NumGLUE-ds  : 0.333
  20Minuten   : 0.000

Stage 3/8: MeetingBank | train loss=3.0741
  C-STANCE    : 1.000
  FOMC        : 0.000
  MeetingBank : 0.000
  Py150       : 0.667
  ScienceQA   : 0.667
  NumGLUE-cm  : 1.000
  NumGLUE-ds  : 0.667
  20Minuten   : 0.000

Stage 4/8: Py150 | train loss=3.4927
  C-STANCE    : 0.667
  FOMC        : 0.000
  MeetingBank : 0.000
  Py150       : 0.667
  ScienceQA   : 0.667
  NumGLUE-cm  : 1.000
  NumGLUE-ds  : 0.333
  20Minuten   : 0.000

Stage 5/8: ScienceQA | train loss=1.8466
  C-STANCE    : 0.667
  FOMC        : 0.000
  MeetingBank : 0.000
  Py150       : 0.667
  Scie

## 4. Compute OP, BWT and forgetting

**OP** is the mean final performance. **BWT** measures how the final performance of earlier tasks differs from their performance immediately after learning them. **Forgetting** is the maximum previous performance minus final performance for each task.

In [13]:
final_row = performance[-1]
OP = float(final_row.mean())

bwt_terms = []
forgetting = []
for j in range(len(task_names)):
    if j < len(task_names)-1:
        bwt_terms.append(performance[-1,j] - performance[j,j])
        forgetting.append(np.max(performance[:j+1,j]) - performance[-1,j])
    else:
        bwt_terms.append(0.0)
        forgetting.append(0.0)
BWT = float(np.mean(bwt_terms))
mean_forgetting = float(np.mean(forgetting))

print(f'OP               = {OP:.6f}')
print(f'BWT              = {BWT:.6f}')
print(f'Mean Forgetting  = {mean_forgetting:.6f}')


OP               = 0.625000
BWT              = 0.000000
Mean Forgetting  = 0.000000


In [14]:
matrix = pd.DataFrame(performance, index=[f'after_{t}' for t in task_names], columns=task_names)
forget_df = pd.DataFrame({'task':task_names, 'forgetting':forgetting})
metrics_df = pd.DataFrame([{'Method':'Qwen2.5-0.5B + Sequential LoRA (CPU baseline)', 'OP':OP, 'BWT':BWT, 'Mean Forgetting':mean_forgetting, 'Tasks':len(task_names)}])

matrix.to_csv(OUT/'performance_matrix.csv')
forget_df.to_csv(OUT/'forgetting.csv', index=False)
metrics_df.to_csv(OUT/'metrics.csv', index=False)
with open(OUT/'manifest.json','w') as f:
    json.dump({'model':MODEL_NAME,'method':'Sequential LoRA','device':'CPU','seed':SEED,'tasks':task_names}, f, indent=2)

display(metrics_df)
display(matrix)
display(forget_df)


,Method,OP,BWT,Mean Forgetting,Tasks
0,Qwen2.5-0.5B + Sequential LoRA (CPU baseline),0.625,0.0,0.0,8


,C-STANCE,FOMC,MeetingBank,Py150,ScienceQA,NumGLUE-cm,NumGLUE-ds,20Minuten
after_C-STANCE,1.000000,0.000000,0.0,0.666667,0.666667,1.0,0.666667,0.000000
after_FOMC,1.000000,0.000000,0.0,0.666667,0.666667,1.0,0.333333,0.000000
after_MeetingBank,1.000000,0.000000,0.0,0.666667,0.666667,1.0,0.666667,0.000000
after_Py150,0.666667,0.000000,0.0,0.666667,0.666667,1.0,0.333333,0.000000
after_ScienceQA,0.666667,0.000000,0.0,0.666667,1.000000,1.0,0.333333,0.000000
after_NumGLUE-cm,0.666667,0.000000,0.0,0.666667,1.000000,1.0,0.333333,0.000000
after_NumGLUE-ds,0.666667,0.333333,0.0,0.666667,1.000000,1.0,0.666667,0.000000
after_20Minuten,1.000000,0.333333,0.0,0.333333,1.000000,1.0,0.666667,0.666667


,task,forgetting
0,C-STANCE,0.000000
1,FOMC,-0.333333
2,MeetingBank,0.000000
3,Py150,0.333333
4,ScienceQA,0.000000
5,NumGLUE-cm,0.000000
6,NumGLUE-ds,0.000000
7,20Minuten,0.000000


## 5. Optional: save the trained adapter

The adapter is small compared with the full model. This cell saves it to the Colab runtime. You can zip/download the results directory if needed.

In [15]:
ADAPTER_DIR = OUT / 'lora_adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved to:', ADAPTER_DIR)
print('Result files:', [p.name for p in OUT.iterdir()])


Saved to: /content/qwen05b_baseline_results/lora_adapter
Result files: ['lora_adapter', 'forgetting.csv', 'manifest.json', 'performance_matrix.csv', 'metrics.csv']


In [17]:
import json
import pandas as pd
from pathlib import Path

OUT = Path('/content/qwen05b_baseline_results')

# 1. Print manifest.json
manifest_path = OUT / 'manifest.json'
if manifest_path.exists():
    print("=== Manifest ===")
    with open(manifest_path, 'r') as f:
        print(json.dumps(json.load(f), indent=2))
    print()

# 2. Print metrics.csv
metrics_path = OUT / 'metrics.csv'
if metrics_path.exists():
    print("=== Metrics ===")
    display(pd.read_csv(metrics_path))
    print()

# 3. Print performance_matrix.csv
perf_path = OUT / 'performance_matrix.csv'
if perf_path.exists():
    print("=== Performance Matrix ===")
    display(pd.read_csv(perf_path, index_col=0))
    print()

# 4. Print forgetting.csv
forget_path = OUT / 'forgetting.csv'
if forget_path.exists():
    print("=== Forgetting ===")
    display(pd.read_csv(forget_path))
    print()

=== Manifest ===
{
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "method": "Sequential LoRA",
  "device": "CPU",
  "seed": 42,
  "tasks": [
    "C-STANCE",
    "FOMC",
    "MeetingBank",
    "Py150",
    "ScienceQA",
    "NumGLUE-cm",
    "NumGLUE-ds",
    "20Minuten"
  ]
}

=== Metrics ===


,Method,OP,BWT,Mean Forgetting,Tasks
0,Qwen2.5-0.5B + Sequential LoRA (CPU baseline),0.625,0.0,0.0,8



=== Performance Matrix ===


,C-STANCE,FOMC,MeetingBank,Py150,ScienceQA,NumGLUE-cm,NumGLUE-ds,20Minuten
after_C-STANCE,1.000000,0.000000,0.0,0.666667,0.666667,1.0,0.666667,0.000000
after_FOMC,1.000000,0.000000,0.0,0.666667,0.666667,1.0,0.333333,0.000000
after_MeetingBank,1.000000,0.000000,0.0,0.666667,0.666667,1.0,0.666667,0.000000
after_Py150,0.666667,0.000000,0.0,0.666667,0.666667,1.0,0.333333,0.000000
after_ScienceQA,0.666667,0.000000,0.0,0.666667,1.000000,1.0,0.333333,0.000000
after_NumGLUE-cm,0.666667,0.000000,0.0,0.666667,1.000000,1.0,0.333333,0.000000
after_NumGLUE-ds,0.666667,0.333333,0.0,0.666667,1.000000,1.0,0.666667,0.000000
after_20Minuten,1.000000,0.333333,0.0,0.333333,1.000000,1.0,0.666667,0.666667



=== Forgetting ===


,task,forgetting
0,C-STANCE,0.000000
1,FOMC,-0.333333
2,MeetingBank,0.000000
3,Py150,0.333333
4,ScienceQA,0.000000
5,NumGLUE-cm,0.000000
6,NumGLUE-ds,0.000000
7,20Minuten,0.000000


## Interpretation for the team

- This is the **baseline**, not FAAR.
- It is intentionally CPU-friendly and uses **Qwen2.5-0.5B-Instruct + LoRA**.
- For the final paper/presentation, compare models only when the **task datasets, splits, prompts, number of training examples, evaluation metric, and training protocol are identical**.
- Do not present this tiny built-in suite as the official TRACE benchmark; replace `TASKS` with the exact benchmark data when an apples-to-apples 0.5B run is required.


### 📊 Analysis of Baseline Sequential LoRA Results

#### 1. Overall Performance & Catastrophic Forgetting
- **Overall Performance (OP): 0.625**
- **Mean Forgetting: 0.000**
- **Backward Transfer (BWT): 0.000**

Generally, sequential fine-tuning of neural networks without regularization leads to catastrophic forgetting, where performance on earlier tasks falls to zero as new tasks are learned. Surprisingly, our baseline shows an average forgetting rate of **0.00**, which is remarkably stable.

#### 2. Key Findings from the Performance Matrix
- **High Stability**: On tasks like **C-STANCE**, **ScienceQA**, and **NumGLUE-cm**, once the model is trained or performs well initially, it maintains that performance till the very end.
- **Minor Catastrophic Forgetting**: The only clear instance of forgetting occurs in **Py150**, where the accuracy drops from **0.67** (at the end of its training stage) to **0.33** in the final stage.
- **Backward Transfer (BWT) & Cross-Task Helpfulness**:
  - Learning **NumGLUE-ds** actually recovered/improved performance on **FOMC** from **0.00** to **0.33**.
  - Learning the final task **20Minuten** restored **C-STANCE**'s accuracy back to **1.00** after it had dipped to **0.67**.

#### 3. Factors Influencing these Numbers
- **Task Representation size**: Because this baseline uses very tiny task samples (3 examples per task), the performance is highly quantized (multiples of $1/3$, like $0.33$, $0.67$, $1.00$).
- **Representation Overlap**: Tasks like **NumGLUE-cm** and **NumGLUE-ds** share strong numerical representations, contributing to transfer/retention.
- **LoRA Capacity**: A rank of $r=4$ acts as a regularizer, restricting the parameter update footprint and naturally dampening dramatic forgetting effects across this tiny sample subset.